### Install necessary libraries
We need `transformers`, `datasets`, and standard deep learning packages to load the CLIP model and the CIFAR-10 dataset.

In [ ]:
!pip install -q transformers datasets torch torchvision

### Load Dataset and Model
We will load a small sample of the CIFAR-10 test split, initialize the CLIP model (`openai/clip-vit-base-patch32`), and define the textual templates for zero-shot classification.

In [ ]:
import torch
from datasets import load_dataset
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
from tqdm import tqdm

# Check GPU availability
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

# Load a subset of CIFAR-10 test set
dataset = load_dataset("cifar10", split="test")
# Shuffle and select 100 images for a quick evaluation demonstration
subset_dataset = dataset.shuffle(seed=42).select(range(100))

# Load CLIP model and processor
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)

# CIFAR-10 classes
class_names = dataset.features["label"].names
print("CIFAR-10 classes:", class_names)

# Create prompt templates
prompts = [f"a photo of a {label}" for label in class_names]

### Run Zero-Shot Classification Inference

We apply the methodology as per original paper - parts 2 and 3 of the image:
![](https://github.com/openai/CLIP/raw/main/CLIP.png)

We collect a sequence of prompts in the style of

`a photo of a {label}`

where `{label}` is replaced by whichever possible description of a class.

In the case of CIFAR10, we will have 10 prompts:

`a photo of a airplane`

`a photo of a automobile`

`a photo of a bird`

`a photo of a cat`

`a photo of a deer`

`a photo of a dog`

`a photo of a frog`

`a photo of a horse`

`a photo of a ship`

`a photo of a truck`

Each of these prompts are projected by the text encoder into the text-image shared embedding space.

We then take an image (belonging to one of these 10 classes) and project it into the embedding space using the visual encoder.

We operate classification by

1. Calculating the euclidean distance between the visual embeddings and each of the 10 text embeddings
2. Selecting as class the one whose text embedding is closest to the image's




In [ ]:
correct = 0
total = len(subset_dataset)

# Put model in evaluation mode
model.eval()

with torch.no_grad():
    for item in tqdm(subset_dataset):
        image = item["img"]
        label_idx = item["label"]

        # Process inputs
        inputs = processor(
            text=prompts,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)

        # Forward pass
        outputs = model(**inputs)

        # Retrieve similarity scores (logits_per_image is image-text similarity)
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]

        # Predict class with highest probability
        predicted_idx = np.argmax(probs)

        if predicted_idx == label_idx:
            correct += 1

accuracy = (correct / total) * 100
print(f"\nZero-shot Accuracy on {total} sample images: {accuracy:.2f}%")

Additional steps to make the classification more accurate:

It has been shown (e.g., Brouwer et al., 2025) that small fine-tuning steps on the VLM can improve classification accuracy, especially in the case of few-shot learning. See repo for code: https://github.com/ericbrouwer0/adaptive-prompt-tuning.


**References**

Brouwer, E., van Woerden, J. E., Burghouts, G., Valdenegro-Toro, M., & Zullich, M. (2024). Adaptive Prompt Tuning: Vision Guided Prompt Tuning with Cross-Attention for Fine-Grained Few-Shot Learning. arXiv preprint arXiv:2412.14640.